In [0]:
%sql
CREATE TABLE IF NOT EXISTS identifier(:catalog || '.gold.daily_kpis') (
    sale_date DATE,
    daily_revenue DECIMAL(14,2),
    daily_orders BIGINT,
    revenue_7day_avg DECIMAL(14,2),
    revenue_30day_avg DECIMAL(14,2),
    running_total_revenue DECIMAL(16,2),
    running_total_orders BIGINT
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW date_dim AS
SELECT explode(sequence(
    (SELECT MIN(sale_date) FROM identifier(:catalog || '.silver.sales_clean')),
    (SELECT MAX(sale_date) FROM identifier(:catalog || '.silver.sales_clean')),
    interval 1 day
)) AS sale_date;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW daily_revenue AS
SELECT
    d.sale_date,
    COALESCE(SUM(s.sale_amount), 0) AS daily_revenue,
    COALESCE(COUNT(s.sale_id), 0)   AS daily_orders
FROM date_dim d
LEFT JOIN identifier(:catalog || '.silver.sales_clean') s ON s.sale_date = d.sale_date
GROUP BY d.sale_date;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW daily_kpis_calc AS
SELECT
    sale_date, daily_revenue, daily_orders,
    ROUND(AVG(daily_revenue) OVER (
        ORDER BY sale_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS revenue_7day_avg,
    ROUND(AVG(daily_revenue) OVER (
        ORDER BY sale_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS revenue_30day_avg,
    SUM(daily_revenue) OVER (
        ORDER BY sale_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_revenue,
    SUM(daily_orders) OVER (
        ORDER BY sale_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_orders
FROM daily_revenue;

In [0]:
%sql
INSERT OVERWRITE identifier(:catalog || '.gold.daily_kpis')
SELECT sale_date, daily_revenue, daily_orders, revenue_7day_avg, revenue_30day_avg,
       running_total_revenue, running_total_orders
FROM daily_kpis_calc;

In [0]:
%sql
SELECT
    curr.sale_date,
    curr.daily_revenue,
    prev.sale_date  AS prior_week_same_weekday,
    prev.daily_revenue AS prior_week_revenue,
    curr.daily_revenue - prev.daily_revenue AS revenue_delta
FROM identifier(:catalog || '.gold.daily_kpis') curr
JOIN identifier(:catalog || '.gold.daily_kpis') prev
    ON prev.sale_date = date_sub(curr.sale_date, 7)
ORDER BY curr.sale_date;